# Project 3: Multi-Agent Web Question Answering System

This project like implements a local multi-agent question answering system with small open-source language models and the system uses two researcher agents, one evaluator agent, and one writer agent to search the web, compare evidence, and produce a final answer with sources you know.

In [1]:
# Install the required packages once in your local environment.
import sys
!{sys.executable} -m pip install -q -U langgraph langchain-ollama ddgs

In [2]:
from __future__ import annotations

import json
import re
import textwrap
import urllib.request
from typing import Any, TypedDict
from urllib.parse import urlparse

from ddgs import DDGS
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph


# The same local model is used for all roles so the notebook stays easy to run.
RESEARCHER_A_MODEL = "llama3.2:3b"
RESEARCHER_B_MODEL = "llama3.2:3b"
EVALUATOR_MODEL = "llama3.2:3b"
WRITER_MODEL = "llama3.2:3b"

RESULTS_PER_RESEARCHER = 4
MAX_FINAL_EVIDENCE = 5


def check_ollama_server() -> None:
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as response:
            payload = json.loads(response.read().decode("utf-8"))
        print("Ollama server is running.")
        print("Available models:")
        for item in payload.get("models", []):
            print(" -", item["name"])
    except Exception as exc:
        print("Could not connect to Ollama at http://localhost:11434")
        print("Please start Ollama locally before running the notebook.")
        print("Details:", exc)


check_ollama_server()

Ollama server is running.
Available models:
 - llava:latest
 - llama3.2-vision:latest
 - llama3.2:3b


In [3]:
researcher_a_llm = ChatOllama(model=RESEARCHER_A_MODEL, temperature=0.1)
researcher_b_llm = ChatOllama(model=RESEARCHER_B_MODEL, temperature=0.1)
evaluator_llm = ChatOllama(model=EVALUATOR_MODEL, temperature=0.0)
writer_llm = ChatOllama(model=WRITER_MODEL, temperature=0.0)
search_client = DDGS()


def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def call_json_agent(llm: ChatOllama, prompt: str, fallback: dict[str, Any]) -> dict[str, Any]:
    try:
        reply = llm.invoke(prompt)
        content = reply.content if hasattr(reply, "content") else str(reply)
        return extract_json_object(content)
    except Exception as exc:
        print(f"JSON fallback used because parsing failed: {exc}")
        return fallback


def normalize_results(raw_results: list[dict[str, Any]]) -> list[dict[str, str]]:
    cleaned = []
    for item in raw_results:
        cleaned.append(
            {
                "title": item.get("title", ""),
                "url": item.get("href", item.get("url", "")),
                "snippet": item.get("body", item.get("snippet", "")),
            }
        )
    return cleaned


def format_evidence(items: list[dict[str, str]]) -> str:
    if not items:
        return "No evidence found."
    blocks = []
    for idx, item in enumerate(items, start=1):
        blocks.append(
            textwrap.dedent(
                f"""
                [{idx}] Title: {item.get('title', 'N/A')}
                URL: {item.get('url', 'N/A')}
                Snippet: {item.get('snippet', 'N/A')}
                """
            ).strip()
        )
    return "\n\n".join(blocks)


def unique_by_url(items: list[dict[str, str]]) -> list[dict[str, str]]:
    seen = set()
    unique_items = []
    for item in items:
        url = item.get("url", "")
        if url and url not in seen:
            seen.add(url)
            unique_items.append(item)
    return unique_items


def simple_search(query: str, max_results: int = RESULTS_PER_RESEARCHER) -> list[dict[str, str]]:
    try:
        raw_results = list(search_client.text(query, max_results=max_results))
        return normalize_results(raw_results)
    except Exception as exc:
        print(f"Search failed for '{query}': {exc}")
        return []


def domain_score(url: str) -> int:
    host = urlparse(url).netloc.lower()
    if any(token in host for token in ["arxiv.org", "google.com", "microsoft.com", "ibm.com", "aws.amazon.com"]):
        return 4
    if any(token in host for token in ["pinecone.io", "techtarget.com", "wikipedia.org"]):
        return 3
    if any(token in host for token in ["blog.", "medium.com"]):
        return 1
    return 2


def heuristic_rank(question: str, items: list[dict[str, str]]) -> list[dict[str, str]]:
    keywords = {word for word in re.findall(r"[a-zA-Z]+", question.lower()) if len(word) > 3}

    def score(item: dict[str, str]) -> tuple[int, int, int]:
        haystack = f"{item.get('title', '')} {item.get('snippet', '')}".lower()
        overlap = sum(1 for word in keywords if word in haystack)
        has_benefit = int(any(word in haystack for word in ["benefit", "advantage", "improv", "pros"]))
        has_limit = int(any(word in haystack for word in ["limit", "challenge", "risk", "tradeoff", "cons"]))
        return (domain_score(item.get('url', '')) + has_benefit + has_limit, overlap, len(item.get('snippet', '')))

    return sorted(items, key=score, reverse=True)


def build_fallback_writer_json(evidence: list[dict[str, str]], confidence: float) -> dict[str, Any]:
    benefits = []
    limitations = []
    for idx, item in enumerate(evidence, start=1):
        text = f"{item.get('title', '')} {item.get('snippet', '')}".lower()
        if len(benefits) < 3 and any(word in text for word in ["benefit", "advantage", "improv", "relevant", "accurate"]):
            benefits.append({"point": item.get("snippet", item.get("title", "")).strip(), "source_index": idx})
        if len(limitations) < 3 and any(word in text for word in ["limit", "challenge", "risk", "tradeoff", "hallucination", "privacy", "complexity"]):
            limitations.append({"point": item.get("snippet", item.get("title", "")).strip(), "source_index": idx})
    if not benefits and evidence:
        benefits.append({"point": evidence[0].get("snippet", evidence[0].get("title", "")).strip(), "source_index": 1})
    if not limitations and len(evidence) > 1:
        limitations.append({"point": evidence[1].get("snippet", evidence[1].get("title", "")).strip(), "source_index": 2})
    return {
        "intro": "The selected sources show that retrieval-augmented generation improves answer grounding, but it still depends heavily on retrieval quality and system design.",
        "benefits": benefits[:3],
        "limitations": limitations[:3],
        "caveat": "The exact strengths and weaknesses depend on the quality of the retriever, the knowledge base, and how the system is evaluated.",
        "confidence": confidence,
    }


def render_final_answer(answer_json: dict[str, Any], evidence: list[dict[str, str]], confidence: float) -> str:
    lines = []
    lines.append("**Answer**")
    lines.append("")
    lines.append(answer_json.get("intro", ""))
    lines.append("")
    lines.append("**Benefits**")
    for item in answer_json.get("benefits", []):
        src_idx = item.get("source_index", 1)
        lines.append(f"- {item.get('point', '').strip()} [Source {src_idx}]")
    lines.append("")
    lines.append("**Limitations**")
    for item in answer_json.get("limitations", []):
        src_idx = item.get("source_index", 1)
        lines.append(f"- {item.get('point', '').strip()} [Source {src_idx}]")
    caveat = answer_json.get("caveat", "")
    if caveat:
        lines.append("")
        lines.append("**Note**")
        lines.append(caveat)
    lines.append("")
    lines.append("**Sources:**")
    for idx, item in enumerate(evidence, start=1):
        lines.append(f"[{idx}] {item.get('url', '')}")
    lines.append("")
    lines.append(f"**Confidence level:** {confidence:.2f}")
    return "\n".join(lines)

In [4]:
class AgentState(TypedDict, total=False):
    query: str
    researcher_a_query: str
    researcher_b_query: str
    researcher_a_results: list[dict[str, str]]
    researcher_b_results: list[dict[str, str]]
    selected_evidence: list[dict[str, str]]
    evaluation_notes: str
    confidence: float
    final_answer: str


def researcher_a_node(state: AgentState) -> AgentState:
    question = state["query"]
    prompt = f"""
You are Researcher Agent 1.
Create one strong web search query for broad background information.

User question: {question}

Return valid JSON only:
{{"search_query": "one search query"}}

Rules:
- Focus on the core concept and main advantages.
- Keep the query short, factual, and useful.
- Do not add markdown fences.
"""
    fallback = {"search_query": question}
    parsed = call_json_agent(researcher_a_llm, prompt, fallback)
    search_query = parsed.get("search_query", question).strip() or question
    return {
        "researcher_a_query": search_query,
        "researcher_a_results": simple_search(search_query),
    }


def researcher_b_node(state: AgentState) -> AgentState:
    question = state["query"]
    prompt = f"""
You are Researcher Agent 2.
Create one strong web search query for limitations, risks, challenges, or tradeoffs.

User question: {question}

Return valid JSON only:
{{"search_query": "one search query"}}

Rules:
- Focus on limitations and failure cases.
- Keep the query short, factual, and useful.
- Do not add markdown fences.
"""
    fallback = {"search_query": f"{question} limitations challenges"}
    parsed = call_json_agent(researcher_b_llm, prompt, fallback)
    search_query = parsed.get("search_query", f"{question} limitations challenges").strip() or f"{question} limitations challenges"
    return {
        "researcher_b_query": search_query,
        "researcher_b_results": simple_search(search_query),
    }


def evaluator_node(state: AgentState) -> AgentState:
    combined = unique_by_url(state.get("researcher_a_results", []) + state.get("researcher_b_results", []))
    combined = heuristic_rank(state["query"], combined)

    prompt = f"""
You are the Evaluator Agent.
Two researchers collected evidence for the same question. Select the strongest items.

Question: {state['query']}
Researcher 1 query: {state.get('researcher_a_query', '')}
Researcher 2 query: {state.get('researcher_b_query', '')}

Evidence:
{format_evidence(combined)}

Return valid JSON only:
{{
  "selected_indices": [1, 2, 3],
  "evaluation_notes": "short explanation",
  "confidence": 0.0
}}

Rules:
- Keep at most {MAX_FINAL_EVIDENCE} items.
- Prefer reliable and relevant sources.
- Prefer evidence that covers both benefits and limitations.
- Confidence must be between 0 and 1.
"""

    fallback = {
        "selected_indices": list(range(1, min(len(combined), MAX_FINAL_EVIDENCE) + 1)),
        "evaluation_notes": "Selected the strongest evidence after combining both researchers and ranking sources.",
        "confidence": 0.75,
    }
    parsed = call_json_agent(evaluator_llm, prompt, fallback)

    selected = []
    for idx in parsed.get("selected_indices", []):
        if isinstance(idx, int) and 1 <= idx <= len(combined):
            selected.append(combined[idx - 1])
    if not selected:
        selected = combined[:MAX_FINAL_EVIDENCE]

    confidence = parsed.get("confidence", 0.75)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.75
    confidence = max(0.0, min(1.0, confidence))

    return {
        "selected_evidence": selected[:MAX_FINAL_EVIDENCE],
        "evaluation_notes": parsed.get("evaluation_notes", fallback["evaluation_notes"]),
        "confidence": confidence,
    }


def writer_node(state: AgentState) -> AgentState:
    evidence = state.get("selected_evidence", [])
    prompt = f"""
You are the Writer Agent.
Use only the selected evidence to answer the question.

Question: {state['query']}
Evaluator notes: {state.get('evaluation_notes', '')}

Selected evidence:
{format_evidence(evidence)}

Return valid JSON only with this schema:
{{
  "intro": "1-2 sentence summary",
  "benefits": [
    {{"point": "benefit supported by the evidence", "source_index": 1}}
  ],
  "limitations": [
    {{"point": "limitation supported by the evidence", "source_index": 2}}
  ],
  "caveat": "optional short note"
}}

Rules:
- Use only claims supported by the evidence.
- Do not turn a limitation into a benefit.
- Keep 2 to 3 benefits and 2 to 3 limitations.
- Use source_index values that point to the selected evidence above.
- Write in clear simple English.
- Do not add markdown fences.
"""

    fallback = build_fallback_writer_json(evidence, state.get("confidence", 0.75))
    parsed = call_json_agent(writer_llm, prompt, fallback)
    final_text = render_final_answer(parsed, evidence, state.get("confidence", 0.75))
    return {"final_answer": final_text}


graph_builder = StateGraph(AgentState)
graph_builder.add_node("researcher_a", researcher_a_node)
graph_builder.add_node("researcher_b", researcher_b_node)
graph_builder.add_node("evaluator", evaluator_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_edge(START, "researcher_a")
graph_builder.add_edge(START, "researcher_b")
graph_builder.add_edge("researcher_a", "evaluator")
graph_builder.add_edge("researcher_b", "evaluator")
graph_builder.add_edge("evaluator", "writer")
graph_builder.add_edge("writer", END)
agent_graph = graph_builder.compile()

In [5]:
def run_multi_agent_qa(question: str) -> AgentState:
    initial_state: AgentState = {"query": question}
    return agent_graph.invoke(initial_state)


def show_run(result: AgentState) -> None:
    print("Question:")
    print(result.get("query", ""))
    print("\nResearcher Agent 1 query:")
    print(" -", result.get("researcher_a_query", ""))
    print("\nResearcher Agent 2 query:")
    print(" -", result.get("researcher_b_query", ""))
    print("\nEvaluator notes:")
    print(result.get("evaluation_notes", ""))
    print("\nConfidence:", result.get("confidence", 0.0))
    print("\nSelected evidence:")
    print(format_evidence(result.get("selected_evidence", [])))
    print("\nFinal answer:\n")
    print(result.get("final_answer", ""))

In [6]:
# Example run.
demo_question = "What are the main benefits and limitations of retrieval-augmented generation (RAG)?"
demo_result = run_multi_agent_qa(demo_question)
show_run(demo_result)

Question:
What are the main benefits and limitations of retrieval-augmented generation (RAG)?

Researcher Agent 1 query:
 - retrieval-augmented generation benefits limitations

Researcher Agent 2 query:
 - retrieval-augmented generation limitations and challenges

Evaluator notes:
Selected items cover both benefits and limitations of RAG. Sources are reliable and relevant.

Confidence: 0.8

Selected evidence:
[1] Title: Retrieval-Augmented Generation: A Comprehensive Survey of
URL: https://arxiv.org/html/2506.00054v1
Snippet: However, integratingretrievalwithgenerationintroduces uniquechallenges:retrievalnoiseandredundancy can degrade output quality; misalignment ...

[2] Title: Retrieval-Augmented Generation (RAG): Overview, Benefits & Limitations
URL: https://www.kiteworks.com/risk-compliance-glossary/retrieval-augmented-generation/
Snippet: September 11, 2025 -These includeimplementation complexity, dependency on external databases for retrieval, security and privacy concerns from a